# Import Libralies

In [1]:
# Math thingy
import numpy as np

# Data Manipulation
import pandas as pd

# Data Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# K-mean Algorithm
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix, adjusted_rand_score

# Read the data file

In [2]:
def read_file(filepath):
    """reads the data file and shows as a dataframe."""
    df = pd.read_csv(filepath)
    return df

In [3]:
iris = 'iris.data'
df = read_file(iris)
df.sample(8)

,5.1,3.5,1.4,0.2,Iris-setosa
74,6.6,3.0,4.4,1.4,Iris-versicolor
132,6.3,2.8,5.1,1.5,Iris-virginica
109,6.5,3.2,5.1,2.0,Iris-virginica
123,6.7,3.3,5.7,2.1,Iris-virginica
58,5.2,2.7,3.9,1.4,Iris-versicolor
27,5.2,3.4,1.4,0.2,Iris-setosa
104,7.6,3.0,6.6,2.1,Iris-virginica
20,5.1,3.7,1.5,0.4,Iris-setosa


# Get overview of the dataset

In [4]:
df.columns

Index(['5.1', '3.5', '1.4', '0.2', 'Iris-setosa'], dtype='object')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149 entries, 0 to 148
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   5.1          149 non-null    float64
 1   3.5          149 non-null    float64
 2   1.4          149 non-null    float64
 3   0.2          149 non-null    float64
 4   Iris-setosa  149 non-null    object 
dtypes: float64(4), object(1)
memory usage: 5.9+ KB


In [6]:
df.shape

(149, 5)

In [7]:
df.isnull().sum().sum()

0

In [8]:
df.duplicated().sum()

3

In [9]:
def preprocessing(filepath):
    """cleans the dataset."""

    mapper = {
        0: "sepal_length",
        1: "sepal_width",
        2: "petal_length",
        3: "petal_width",
        4: "species"
    }

    # Read the data file and ADD header=None
    df = pd.read_csv(filepath, header=None)
    
    # drop duplicated rows 
    df = df.drop_duplicates()
    
    # Reset the index after dropping rows
    df = df.reset_index(drop=True) # drop=True prevents the old index from becoming a new column

    # Rename the columns
    df = df.rename(columns=mapper)

    return df

In [10]:
df_iris = preprocessing(iris)
df_iris.sample(5)

,sepal_length,sepal_width,petal_length,petal_width,species
102,6.5,3.0,5.8,2.2,Iris-virginica
68,5.9,3.2,4.8,1.8,Iris-versicolor
81,6.0,2.7,5.1,1.6,Iris-versicolor
24,4.8,3.4,1.9,0.2,Iris-setosa
129,7.9,3.8,6.4,2.0,Iris-virginica


# K-means Algorithm

In [11]:
# Assuming df_iris is correctly loaded and cleaned (147 rows)
# ===================================================================
# DATA PREPARATION: Standardize and PCA
# ===================================================================

# 1. Select only the numerical features for clustering
X = df_iris[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]

# 2. Standardize the data (Correctly using StandardScaler)
scaler = StandardScaler() 
X_scaled = scaler.fit_transform(X)

# 3. Initialize PCA to keep 3 principal components
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled) # X_pca is the final feature set

explained_variance = pca.explained_variance_ratio_.sum()
print(f"Total Variance Explained by 3 Components: {explained_variance:.4f}") # Should be 0.9935

# ===================================================================
# K-MEANS CLUSTERING (Using the best data: X_pca)
# ===================================================================

# Initialize and Fit the model to the PCA-transformed data
kmeans_pca = KMeans(n_clusters=3, random_state=42, n_init='auto')
kmeans_pca.fit(X_pca)

# Get the cluster labels
df_iris['Cluster_PCA'] = kmeans_pca.labels_ 

# ===================================================================
# MODEL EVALUATION (Using the Cluster_PCA column)
# ===================================================================

# True labels
species_mapping = {'Iris-setosa': 0, 'Iris-versicolor': 1, 'Iris-virginica': 2}
y_true = df_iris['species'].map(species_mapping)

# Predicted labels
y_pred_pca = df_iris['Cluster_PCA'] 

# Calculate the Adjusted Rand Index
ari_score_pca = adjusted_rand_score(y_true, y_pred_pca)

print(f"\nFINAL Adjusted Rand Index (ARI) Score: {ari_score_pca:.4f}")
print("FINAL Confusion Matrix (True vs. Predicted Clusters):")
print(confusion_matrix(y_true, y_pred_pca))

Total Variance Explained by 3 Components: 0.9947

FINAL Adjusted Rand Index (ARI) Score: 0.6479
FINAL Confusion Matrix (True vs. Predicted Clusters):
[[ 1  0 47]
 [37 13  0]
 [ 7 42  0]]
